# 생성 → 검출 검증 — 원문제 모범답안의 Magic 패턴 그대로 (Colab)

**연습 기법** (IOAI 2024 *onsite* **Madarian Cow** 모범답안과 동일): **동결된 생성기 + 동결된 검출기**는 그대로 두고,
작은 **`Magic` 레이어**만 학습해 *생성기의 잠재(latent)를 조작* → 검출기가 **목표 클래스로 검출**하게 만든다.

| Madarian Cow 모범답안 | 이 노트북 (MNIST, 경량) |
|---|---|
| 동결 **Stable Diffusion** (생성) | 동결 **디코더**(잠재 z→숫자 이미지) |
| 동결 **DETR** (검출) | 동결 **분류기**(숫자 검출) |
| **`class Magic(nn.Module)`** — latent 조작 | 동일 (`Magic`: 목표→latent) |
| Magic 을 최적화해 DETR 가 *소(cow)* 검출 | Magic 을 최적화해 분류기가 *목표 숫자* 검출 |
| `detect()` 로 검증 | 분류기 argmax 로 검증 |

**왜 MNIST 인가**: Madarian Cow 모범답안은 *Stable Diffusion + DETR*(GB급, IOAI 중 가장 무거움)이라 Colab 기본 연습엔
부적합. 여기선 **같은 메커니즘**(동결 생성기+동결 검출기에 대해 Magic 최적화)을 28×28 숫자로 1~2분에 익힌다.

> ⚙️ GPU 권장(작은 모델, T4 ~2분).  ⚠️ API 토큰 평문 — 외부 공유 금지. (08·13·18 과 같은 digit-recognizer)

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 에서 Digit Recognizer 다운로드 & 로딩

In [ ]:
import glob, zipfile, pandas as pd, torch.nn as nn, torch.nn.functional as F
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
device = "cuda" if torch.cuda.is_available() else "cpu"
train = pd.read_csv(os.path.join(WORK_DIR, "train.csv")); test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
X = torch.tensor(train.drop(columns=["label"]).values, dtype=torch.float32).div(255.).view(-1, 1, 28, 28).to(device)
y = torch.tensor(train["label"].values, dtype=torch.long).to(device)
X_test = torch.tensor(test.values, dtype=torch.float32).div(255.).view(-1, 1, 28, 28).to(device)
print("train:", X.shape, "| test:", X_test.shape)

## 2. 검출기 (DETR 역할) — 학습 후 **동결**
생성 이미지가 목표 숫자인지 판정. Madarian Cow 의 DETR 처럼 *학습하지 않고 고정*해 둔다.

In [ ]:
detector = nn.Sequential(
    nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32*7*7, 10)).to(device)
opt = torch.optim.Adam(detector.parameters(), 1e-3)
for ep in range(3):
    perm = torch.randperm(len(X))
    for i in range(0, len(X), 256):
        b = perm[i:i+256]; opt.zero_grad(); F.cross_entropy(detector(X[b]), y[b]).backward(); opt.step()
for p in detector.parameters(): p.requires_grad = False    # 동결 (DETR 처럼)
detector.eval()
print("검출기 학습+동결 완료 — train acc:", round((detector(X).argmax(1)==y).float().mean().item(), 3))

## 3. 생성기 (디퓨전 역할) — 오토인코더 디코더, 학습 후 **동결**
잠재 `z`(16차원) → 숫자 이미지. Madarian Cow 의 Stable Diffusion 처럼 *고정된 생성기*. 클래스 입력은 없다
(그래서 Magic 이 잠재를 조작해 목표 숫자를 만들어야 한다).

In [ ]:
ZD = 16
class Enc(nn.Module):
    def __init__(s): super().__init__(); s.f = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(), nn.Linear(256,ZD))
    def forward(s, x): return s.f(x)
class Dec(nn.Module):                                       # 생성기 (디퓨전 역할): z -> 이미지
    def __init__(s): super().__init__(); s.f = nn.Sequential(nn.Linear(ZD,256), nn.ReLU(), nn.Linear(256,784), nn.Sigmoid())
    def forward(s, z): return s.f(z).view(-1, 1, 28, 28)

enc, gen = Enc().to(device), Dec().to(device)
ae = torch.optim.Adam(list(enc.parameters()) + list(gen.parameters()), 1e-3)
for ep in range(10):
    perm = torch.randperm(len(X))
    for i in range(0, len(X), 256):
        b = perm[i:i+256]; ae.zero_grad(); F.mse_loss(gen(enc(X[b])), X[b]).backward(); ae.step()
for p in gen.parameters(): p.requires_grad = False         # 동결 (디퓨전처럼)
gen.eval()
print("생성기(디코더) 학습+동결 완료")

## 4. Magic 레이어 (모범답안의 `class Magic`) — 동결 생성기+검출기에 대해 최적화
Madarian Cow 의 `Magic` 은 디퓨전 latent 를 조작해 DETR 가 소를 검출하게 만든다. 여기선 `Magic(목표)→z` 를 학습해
**동결 생성기**의 출력이 **동결 검출기**에서 목표 숫자로 나오게 한다 — 손실 = `CE(detector(gen(Magic(목표))), 목표)`.

In [ ]:
class Magic(nn.Module):                                     # 목표 클래스(one-hot) → 생성기 latent 조작
    def __init__(s): super().__init__(); s.f = nn.Sequential(nn.Linear(10,64), nn.ReLU(), nn.Linear(64,ZD))
    def forward(s, target_onehot): return s.f(target_onehot)

magic = Magic().to(device); mo = torch.optim.Adam(magic.parameters(), 1e-2)
targets = torch.eye(10, device=device)                      # 0~9 one-hot
for it in range(400):
    c = targets[torch.randint(0, 10, (128,), device=device)]
    logits = detector(gen(magic(c)))                        # 동결 생성기→동결 검출기 통과
    mo.zero_grad(); F.cross_entropy(logits, c.argmax(1)).backward(); mo.step()   # Magic 만 학습
print("Magic 최적화 완료 (동결 생성기+검출기에 대해)")

## 5. 생성 → 검출 → 검증 (Madarian Cow 패턴)
각 목표 숫자에 대해 `gen(magic(목표))` 로 생성하고, 검출기가 목표로 검출하는 비율을 본다(높을수록 좋음).

In [ ]:
import matplotlib.pyplot as plt
magic.eval()
with torch.no_grad():
    imgs = gen(magic(targets))                              # 10개 목표 숫자 생성
    pred = detector(imgs).argmax(1).cpu().numpy()
acc = (pred == np.arange(10)).mean()
print("검출 검증 정확도(생성→검출, 목표 일치) =", round(float(acc), 3), "| 검출 결과:", pred.tolist())
fig, ax = plt.subplots(1, 10, figsize=(12, 1.5))
for d in range(10):
    ax[d].imshow(imgs[d,0].cpu(), cmap="gray"); ax[d].axis("off"); ax[d].set_title(f"{d}\u2192{pred[d]}")
plt.suptitle("Magic 으로 동결 생성기를 조작해 목표 숫자 생성 (Madarian Cow 패턴)"); plt.show()

## 6. (보너스) Digit Recognizer 제출 — 검출기로 test 예측
검출기(분류기)로 실제 test 를 예측해 `submission.csv` 생성(상시 제출 가능).

In [ ]:
import numpy as np
with torch.no_grad():
    pred = detector(X_test).argmax(1).cpu().numpy()
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"ImageId": np.arange(1, len(pred)+1), "Label": pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path); pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 정리
Madarian Cow **모범답안** 메커니즘(**동결 생성기 + 동결 검출기 + 학습되는 `Magic` 레이어로 latent 조작 → 검출기가
목표 검출**)을 MNIST 로 그대로 옮겼다. Stable Diffusion·DETR 대신 경량 디코더·분류기를 써서 1~2분에 같은 흐름을
익힌다. 더: 생성기를 디퓨전/GAN 으로, 검출기를 다중객체(소+소화전 같은 *조합 제약*)로 확장. **보너스** `submission.csv`
는 [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 에 상시 제출 가능.